## Preparation

First, we will pull the lesson pages straight from the course repository. We will use the commit 8c1834d to make sure everyone works with the exact same data.

We will use gitsource for that:

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

GithubRepositoryDataReader downloads the entire repository and goes over all the files in it. Because we specify allowed_extensions={"md"}, it only checks the markdown files.

We also pass a filename_filter so we don't grab every markdown file in the repo, like the top-level README. The lesson pages all live under a module's lessons/ folder, so filtering on /lessons/ keeps just those.

Each file has a parse() method that returns a dictionary with its filename and content:

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

## Q1. How many lesson pages

How many lesson pages are in the dataset?

In [1]:
len(documents)

NameError: name 'documents' is not defined

## Q2. Indexing and searching

Index the documents with minsearch - make `content` a text field and
`filename` a keyword field. Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

In [4]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [5]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

## Q3. RAG

Now we will build a RAG assistant on top of this data. Let's use the rag helper 
script we prepared during the lessons:

```bash
wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
```

`RAGBase` was written for the FAQ schema (`section`/`question`/`answer`),
while our documents have `filename` and `content`.


Two solutions are possible:

- Implement the RAG flow yourself
- Take `RAGBase` and change the parts related to the FAQ schema - `search` (to use our index) and `build_context`

Build a RAG over the index from Q2 and answer the query:

> How does the agentic loop keep calling the model until it stops?

Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for
this request?

In [6]:
import os 
from rag_helper import RAGBase
from google import genai

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client,
)

answer, tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print(tokens)

Tokens - Input: 7933, Output: 352
The agentic loop keeps calling the model until it stops by wrapping the interaction in a `while` loop.

Here's how it works:
1.  **Initial Call:** The process starts by sending the user's question and the agent's instructions to the model.
2.  **Model's Response:** The model processes the input and returns a response.
3.  **Check for Function Calls:** The agentic loop checks if the model's response contains any `function_call` entries. A `has_function_calls` flag is used for this.
4.  **Execute Tools and Continue Loop:**
    *   If the model's response includes a function call (e.g., `search`), the agent executes that function (like calling the `make_call` helper).
    *   The output of the function call is then appended to the message history.
    *   Since `has_function_calls` is `True`, the loop continues, and the entire updated message history (including the original question, the model's function call, and the tool's output) is sent back to the mo


## Q4. Chunking

The lesson pages are long - some are thousands of characters. Long documents
make retrieval less precise: a match deep inside a page still pulls in the
whole page. A common fix is chunking: split each page into smaller,
overlapping pieces and index those instead.

gitsource has a helper for this: `chunk_documents`. It uses a sliding
window - a window of `size` characters slides across the text in steps of
`step` characters, and each window position becomes one chunk:

```python
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
```

With `size=2000` and `step=1000` (you can see the implementation
[here](https://github.com/alexeygrigorev/gitsource/blob/master/gitsource/chunking.py)):

- Each chunk is a window of `size` characters of the page.
- The window moves forward by `step` characters between chunks. Since `step`
  is smaller than `size`, consecutive chunks overlap by `size - step` (1000)
  characters, so a passage split across a boundary still appears whole in one
  of the chunks.
- Every chunk keeps the original fields (`filename`) and adds `start` (the
  offset in the page) and `content` (the chunk text).

How many chunks do you get?

In [7]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [8]:
len(chunks)

295

## Q5. RAG with chunking

Chunking makes each request smaller, because we send a smaller context to the
LLM. Let's measure that.

Index the chunks from Q4 (same as before: `content` as a text field,
`filename` as a keyword field), point your RAG at the chunk index, and
answer the same query again - reading the input tokens the same way as in Q3.

Compare the input tokens with Q3. How many fewer input tokens does the chunked
version send?


In [9]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(chunks)

In [10]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

assistant = RAGBase(
    index=index,
    llm_client=client,
)

answer, tokens = assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(answer)
print(tokens)

Tokens - Input: 2585, Output: 338
The agentic loop keeps calling the model until it stops through the following mechanism:

1.  **`while True` loop:** The core of the agent is an infinite `while True` loop that continuously calls the model.
2.  **Model's Decision:** In each iteration, the model (`openai_client.responses.create`) is called with the current `messages` history, which includes previous prompts, model outputs, and any tool results. The model then reasons about the next action.
3.  **Checking for Function Calls:** After the model returns a `response`, the code iterates through `response.output`. It sets a `has_function_calls` flag to `True` if any `item.type == "function_call"` is found.
4.  **Executing Tools and Updating History:** If function calls are detected, they are executed (`make_call`), and their `call_output` (results) are appended to the `messages` list. This updated `messages` list is then sent back to the model in the next iteration.
5.  **Exit Condition:** The

## Q6. Turning it into an agent

So far search runs once, with the exact query. Let's make it agentic: give
the LLM a `search` tool and let it decide when (and what) to search. We
suggest [toyaikit](https://github.com/alexeygrigorev/toyaikit), the small
agent library from the module, but you can use anything you like - the OpenAI
Agents SDK, PydanticAI, LangChain, or a hand-written loop.


Create a `search` function that uses the chunk index. Give it a type hint and
a docstring - most frameworks read them to build the tool schema for you.

Build an agent with your `search` tool and run it (with toyaikit, the same way
as in the ToyAIKit lesson). Use these instructions for the agent (they nudge
it to search a few times):

> You're a course teaching assistant. Answer the student's question using the
> search tool. Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

The agent decides on its own when to search and when to answer. Count how many
times it called the `search` tool.

How many times did the agent call `search`?

In [11]:
instructions = '''
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.'''

In [12]:
from langchain_core.tools import tool

@tool
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
    )

In [13]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "gemini-2.5-flash",
    model_provider="google-genai",
    temperature=0.5,
    timeout=600,
    max_tokens=25000,
    streaming=True,
)

In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[search],
    system_prompt=instructions,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "How does the agentic loop work, and how is it different from plain RAG?"}]}
)
print(result["messages"][-1].text)

history = result.get("intermediate_steps", [])

# Since 'intermediate_steps' tracks every action taken:
total_calls = len(history)
print(total_calls)

The agentic loop and plain RAG (Retrieval Augmented Generation) both involve using Large Language Models (LLMs) to answer questions, but they differ significantly in their approach to control and flexibility.

**Plain RAG** operates on a fixed, three-step pipeline:
1.  **Search:** It first performs a search (e.g., keyword or vector search) based on the user's question to retrieve relevant documents or information from a knowledge base.
2.  **Build Prompt:** The retrieved information is then combined with the original question to construct a comprehensive prompt.
3.  **Generate Answer:** This prompt is fed into the LLM, which then generates an answer grounded in the provided context.

The main characteristic of plain RAG is its **predetermined and rigid workflow**. The steps are always executed in the same sequence, regardless of the quality of the search results. If the initial search fails to find relevant information (e.g., due to typos in the query or the need for information from m